# Normalize and classify aspect terms

This notebook is part of the public reproducibility workflow for the study.

Public workflow stage: 04  
Purpose: Normalize extracted terms and apply the frozen 12-class taxonomy.

Source notebooks:
- `clustering.ipynb`: formal term cleaning and normalization rules
- `clustering_n_rev.ipynb`: final human-corrected clean mapping lineage
- `preprocessed_nc/aspect_dataset/cluster_outputs/LLM_clustering.ipynb`: optional-method lineage only
- `主表及分析表构建.ipynb`: final normalized-count join and coverage logic (cell 8)

Input: `data/work/review_term_counts.parquet`; demo mapping `data/sample/sample_term_mapping.csv`, or authorized frozen mapping `data/taxonomy/term_class_mapping.csv`.  
Outputs: `data/work/review_term_counts_normalized.parquet` and `data/work/mapping_coverage.csv`.

The frozen map is the final human-corrected `all_terms_final_postrerun_clean.csv` lineage. No real mapping data is bundled here. `casefixed` and `counts_fixed` are not separate taxonomies.


In [ ]:
from pathlib import Path
import sys

_places = (Path.cwd().resolve(), Path.cwd().resolve().parent)
_root = next((p for p in _places if (p / "src" / "project_paths.py").is_file()), None)
if _root is None:
    raise RuntimeError("Start Jupyter from the public repository root or notebooks directory")
sys.path.insert(0, str(_root))
from src.project_paths import CONFIG_DIR, project_path

try:
    import yaml
except ImportError as exc:
    raise ImportError("PyYAML is required to read the example configuration") from exc
with (CONFIG_DIR / "config.example.yaml").open(encoding="utf-8") as _stream:
    config = yaml.safe_load(_stream)
sample_dir = project_path(config["paths"]["sample_data"])
work_dir = project_path(config["paths"]["work_data"])

import re
import numpy as np
import pandas as pd

ASPECT_CLASSES = [
    "Methodology", "Results", "Experiments and Evaluation", "Data and Datasets",
    "Figures and Tables", "Writing, Language and Terminology", "Theory and Models",
    "Comparison and Context", "Paper Assessment", "Novelty and Impact",
    "Validity, Rigor and Quality", "Limitations",
]


mode = config["taxonomy"]["mode"]
if mode not in {"demo", "formal"}:
    raise ValueError("taxonomy.mode must be 'demo' or 'formal'")
mapping_key = "demo_mapping_file" if mode == "demo" else "mapping_file"
mapping_path = project_path(config["taxonomy"][mapping_key])
counts_path = work_dir / "review_term_counts.parquet"
for path in (counts_path, mapping_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required input is not available yet: {path}")
work_dir.mkdir(parents=True, exist_ok=True)


## Formal normalization rules

The source applies lowercasing, separator and punctuation cleaning, a fixed token map, optional locally available WordNet noun lemmatization, and one narrowly scoped phrase reorder. A missing local WordNet corpus does not trigger a download; the source's no-WordNet fallback is used.


In [ ]:
EVAL_TERMS = {
    "clarity", "interpretation", "validity", "completeness", "novelty", "correctness",
    "robustness", "comparison", "reproducibility", "design", "report", "limitation",
    "usefulness", "focus", "description", "detail", "evidence", "consistency",
    "significance", "organization", "quality", "justification", "applicability",
    "grammar", "importance", "availability", "relevance", "evaluation",
    "presentation", "utility", "suggestion", "improvement", "suitability",
    "rationale", "assumption", "impact", "advantage", "explanation",
    "recommendation", "discussion", "feasibility", "specificity", "accessibility",
    "performance", "scalability", "implication", "reference",
}
WEAK_REVIEW_HEADS = {"application", "analysis", "work"}
REVIEW_TARGETS = {
    "method", "result", "writing", "data", "figure", "table", "theory", "experiment",
    "terminology", "analysis", "supplementary", "presentation", "organization",
    "study", "publication", "related", "model", "approach", "dataset",
}
FORCE_TERM_TYPE = {
    "related work": "review_aspect", "data analysis": "review_aspect",
    "method application": "review_aspect", "method evaluation": "review_aspect",
    "data presentation": "review_aspect", "method utility": "review_aspect",
    "method suggestion": "review_aspect", "publication recommendation": "review_aspect",
    "publication suitability": "review_aspect",
}
TOKEN_MAP = {
    "methods": "method", "results": "result", "experiments": "experiment",
    "datasets": "dataset", "figures": "figure", "tables": "table",
    "studies": "study", "materials": "material", "details": "detail",
    "descriptions": "description", "comparisons": "comparison",
    "limitations": "limitation", "publications": "publication",
    "recommendations": "recommendation", "assumptions": "assumption",
    "advantages": "advantage", "impacts": "impact", "discussions": "discussion",
    "implications": "implication", "references": "reference",
    "experimental": "experiment", "reporting": "report", "validation": "validity",
    "interpretability": "interpretation", "supplemental": "supplementary",
    "statistical": "statistics",
}
STRONG_DOMAIN_HEADS = {
    "factor", "ratio", "tool", "assay", "marker", "sequencing", "seq", "reference",
    "spectrum", "peak", "imaging", "gene", "protein", "rna", "dna", "roi",
    "xrd", "pcr", "sirna", "mrna",
}
_LEMMATIZER = None
_WORDNET_OK = False
try:
    from nltk.stem import WordNetLemmatizer
    _LEMMATIZER = WordNetLemmatizer()
    _LEMMATIZER.lemmatize("methods", pos="n")
    _WORDNET_OK = True
except (ImportError, LookupError):
    print("Local WordNet corpus unavailable; using the source no-WordNet fallback")

def basic_clean(text):
    rules = []
    if pd.isna(text):
        return "", rules
    raw = str(text)
    value = raw.strip()
    if value != raw:
        rules.append("strip")
    lower = value.lower()
    if lower != value:
        rules.append("lower")
    value = lower
    clean = re.sub(r"[/_\-]+", " ", value)
    if clean != value:
        rules.append("separator_to_space")
    value = clean
    clean = re.sub(r"[^a-z0-9\s]", " ", value)
    if clean != value:
        rules.append("punct_remove")
    value = clean
    clean = re.sub(r"\s+", " ", value).strip()
    if clean != value:
        rules.append("space_normalize")
    return clean, rules

def normalize_token(token):
    if token in TOKEN_MAP:
        mapped = TOKEN_MAP[token]
        return mapped, [f"token_map:{token}->{mapped}"] if mapped != token else []
    if _WORDNET_OK and token.isalpha():
        try:
            lemma = _LEMMATIZER.lemmatize(token, pos="n")
            if lemma != token:
                return lemma, [f"wordnet_noun:{token}->{lemma}"]
        except LookupError:
            pass
    return token, []

def normalize_phrase(term_raw):
    term_clean, rules = basic_clean(term_raw)
    if not term_clean:
        return "", "", ""
    normalized, token_rules = [], []
    for token in term_clean.split():
        result, applied = normalize_token(token)
        normalized.append(result)
        token_rules.extend(applied)
    if len(normalized) == 3 and normalized[1] == "of" and normalized[0] in EVAL_TERMS:
        normalized = [normalized[2], normalized[0]]
        token_rules.append("phrase_reorder:eval_of_target")
    return term_clean, " ".join(normalized).strip(), "; ".join(rules + token_rules) or "identity"

def is_strong_domain_term(term_raw, tokens):
    raw = str(term_raw)
    return bool(re.search(r"\b[A-Z]{2,}(?:-[A-Za-z0-9]+)?\b", raw)
                or re.search(r"\b[A-Za-z]+\d+[A-Za-z0-9]*\b", raw)
                or ("/" in raw and re.search(r"[A-Z]{2,}", raw))
                or len(re.findall(r"\b[A-Z]{2,}\b", raw)) >= 2
                or (any(token in STRONG_DOMAIN_HEADS for token in tokens)
                    and (re.search(r"[A-Z]{2,}", raw) or re.search(r"\d", raw))))

def classify_term_type(term_raw, term_norm):
    if not term_norm:
        return "mixed_or_unclear"
    if term_norm in FORCE_TERM_TYPE:
        return FORCE_TERM_TYPE[term_norm]
    tokens = term_norm.split()
    if len(tokens) == 1:
        return "singleton"
    n_eval = sum(token in EVAL_TERMS for token in tokens)
    if is_strong_domain_term(term_raw, tokens):
        return "mixed_or_unclear" if n_eval >= 2 or len(tokens) >= 4 else "domain_term"
    if n_eval >= 2:
        return "mixed_or_unclear"
    if tokens[0] in REVIEW_TARGETS and tokens[-1] in WEAK_REVIEW_HEADS:
        return "review_aspect"
    if tokens[0] == "related" and tokens[-1] == "work":
        return "review_aspect"
    return "review_aspect" if 2 <= len(tokens) <= 4 else "mixed_or_unclear"


## Apply the frozen mapping and calculate coverage

The taxonomy table is read only. The demo file is synthetic and has the same final-class contract. Duplicate normalized keys with conflicting classes are rejected; unmatched terms remain unclassified and are omitted from downstream class counts, as in the formal source.


In [ ]:
review_term_counts = pd.read_parquet(counts_path)
required = {"review_id", "paper_code", "term_raw", "count_in_review", "n_sent_hit"}
if missing := required.difference(review_term_counts.columns):
    raise KeyError(f"Review-term input is missing columns: {sorted(missing)}")
review_term_counts = review_term_counts.copy()
review_term_counts["term_raw"] = review_term_counts["term_raw"].astype(str).str.strip()
review_term_counts["term_raw_key"] = review_term_counts["term_raw"].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
unique_terms = pd.DataFrame({"term_raw": sorted(review_term_counts["term_raw"].dropna().unique())})
normalized = unique_terms["term_raw"].map(normalize_phrase)
unique_terms["term_clean"] = [entry[0] for entry in normalized]
unique_terms["term_norm"] = [entry[1] for entry in normalized]
unique_terms["norm_rule"] = [entry[2] for entry in normalized]
unique_terms["term_type"] = unique_terms.apply(
    lambda row: classify_term_type(row["term_raw"], row["term_norm"]), axis=1)
unique_terms["term_raw_key"] = unique_terms["term_raw"].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
unique_terms["term_norm_key"] = unique_terms["term_norm"].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
unique_terms = unique_terms.drop_duplicates("term_raw_key", keep="first")

final_terms = pd.read_csv(mapping_path, encoding="utf-8-sig")
if missing := {"term_norm", "major_class_final"}.difference(final_terms.columns):
    raise KeyError(f"Frozen mapping is missing columns: {sorted(missing)}")
final_terms = final_terms.copy()
final_terms["term_norm_key"] = final_terms["term_norm"].astype(str).str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
final_terms["major_class_final"] = final_terms["major_class_final"].astype("string").str.strip()
invalid = set(final_terms["major_class_final"].dropna()) - set(ASPECT_CLASSES)
if invalid:
    raise ValueError(f"Frozen mapping contains classes outside the formal taxonomy: {sorted(invalid)}")
conflicts = final_terms.groupby("term_norm_key")["major_class_final"].nunique(dropna=True)
if (conflicts > 1).any():
    raise ValueError("Frozen mapping has conflicting labels for one normalized key")
final_terms = final_terms.drop_duplicates("term_norm_key", keep="first")
review_term_norm = review_term_counts.merge(
    unique_terms[["term_raw_key", "term_clean", "term_norm", "term_norm_key", "norm_rule", "term_type"]],
    on="term_raw_key", how="left", validate="many_to_one")
class_cols = ["term_norm_key", "major_class_final"]
if "label_source_final" in final_terms:
    class_cols.append("label_source_final")
review_term_norm = review_term_norm.merge(
    final_terms[class_cols], on="term_norm_key", how="left", validate="many_to_one")
for column in ("count_in_review", "n_sent_hit"):
    review_term_norm[column] = pd.to_numeric(review_term_norm[column], errors="coerce").fillna(0)
review_term_norm.to_parquet(work_dir / "review_term_counts_normalized.parquet", index=False)

total_rows = len(review_term_norm)
total_count = review_term_norm["count_in_review"].sum()
mapped_norm = review_term_norm["term_norm"].notna() & review_term_norm["term_norm"].astype(str).str.len().gt(0)
mapped_class = review_term_norm["major_class_final"].notna() & review_term_norm["major_class_final"].astype(str).str.len().gt(0)
coverage = pd.DataFrame({"item": [
    "total_review_term_rows", "total_term_count", "rows_mapped_to_term_norm",
    "rows_mapped_to_final_class", "row_norm_coverage", "row_class_coverage",
    "count_mapped_to_term_norm", "count_mapped_to_final_class",
    "count_norm_coverage", "count_class_coverage"],
    "value": [total_rows, total_count, int(mapped_norm.sum()), int(mapped_class.sum()),
              mapped_norm.mean(), mapped_class.mean(),
              review_term_norm.loc[mapped_norm, "count_in_review"].sum(),
              review_term_norm.loc[mapped_class, "count_in_review"].sum(),
              review_term_norm.loc[mapped_norm, "count_in_review"].sum() / total_count if total_count > 0 else np.nan,
              review_term_norm.loc[mapped_class, "count_in_review"].sum() / total_count if total_count > 0 else np.nan]})
coverage.to_csv(work_dir / "mapping_coverage.csv", index=False)
print(f"Mapped {int(mapped_class.sum())} of {total_rows} review-term rows to frozen classes")


## Optional methodological alternatives

BGE and MiniLM embeddings, LLM clustering, and manual review contributed to the historical development of the frozen mapping. They are **not required to reproduce the frozen study analysis** and are intentionally not run or downloaded here. The public workflow consumes the reviewed mapping artifact; it does not regenerate it.
